# Recherche par le sens : embeddings + base vectorielle

**Objectif** : découper les documents, calculer les embeddings, monter la base vectorielle, puis comparer recherche par mots-clés et recherche sémantique sur nos questions.

| Étape | Où |
|---|---|
| 1. Découper les PDF en passages (chunks) | `src/build_chunks.py` → `data/processed/chunks.jsonl` |
| 2. Calculer les embeddings + monter la base vectorielle | **ce notebook** |
| 3. Comparer mots-clés / sémantique sur les questions | `python src/compare_search.py` → `docs/evaluation/comparaison_recherche.md` |


## Préparation

En local : `pip install -r requirements.txt` une seule fois, depuis la racine du projet.

In [12]:
import sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "rag_config.py").exists()), None)
if ROOT is None:
    # Recherche locale dans le dossier extrait si le CWD ne permet pas de remonter directement
    config_paths = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        try:
            config_paths.extend(base.rglob("src/rag_config.py"))
        except PermissionError:
            pass

    if config_paths:
        ROOT = config_paths[0].parent.parent

    if ROOT is None:
        raise FileNotFoundError(
            "Extrayez puis ouvrez le notebook depuis le dossier COGNIVEX "
            "(C:\\...\\COGNIVEX\\notebooks\\recherche.ipynb)."
        )
sys.path.insert(0, str(ROOT / "src"))
print("Racine du projet :", ROOT)

from rag_config import (COLLECTION_NAME, EMBEDDING_MODEL, TOP_K, VECTOR_DB_DIR,
                        charger_chunks, passage_prefix, query_prefix, texte_indexe)
print("Modèle d'embeddings :", EMBEDDING_MODEL)

Racine du projet : c:\Users\LENOVO\Downloads\COGNIVEX-GIT
Modèle d'embeddings : intfloat/multilingual-e5-small


## 1. Charger les chunks

In [13]:
from collections import Counter

chunks = charger_chunks()
print(f"{len(chunks)} chunks chargés\n")
for dci, n in sorted(Counter(c["dci"] for c in chunks).items()):
    print(f"{dci:40s} {n:5d} chunks")

4411 chunks chargés

ACIDE-ACETYLSALICYLIQUE                     55 chunks
AMLODIPINE                                 204 chunks
AMOXICILLINE                               526 chunks
AMOXICILLINE + ACIDE CLAVULANIQUE          199 chunks
ATÉNOLOL                                    64 chunks
AZITHROMYCINE                               38 chunks
BISOPROLOL                                  63 chunks
CEFTRIAXONE                                118 chunks
CIPROFLOXACINE                              49 chunks
CLARITHROMYCINE                             32 chunks
CLINDAMYCINE                                93 chunks
COTRIMOXAZOLE                               88 chunks
CÉFIXIME                                   398 chunks
CÉFUROXIME                                 192 chunks
CÉLÉCOXIB                                  142 chunks
DICLOFENAC                                 273 chunks
DOXYCYCLINE                                 34 chunks
GENTAMICINE                                165 chunks
IBUPROF

## 2. Calculer les embeddings

Un **embedding** transforme un texte en une liste de nombres (ici 384) : deux textes de sens proche donnent des vecteurs proches. Le modèle `multilingual-e5-small` comprend le français et accepte 512 tokens (nos chunks de 120 mots passent sans être tronqués). Il demande le préfixe `passage: ` côté documents et `query: ` côté questions.

> La 1re exécution télécharge le modèle (~470 Mo, internet nécessaire).

In [15]:
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

modele = SentenceTransformer(EMBEDDING_MODEL)

Note: you may need to restart the kernel to use updated packages.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1052.14it/s]


In [16]:
print("Modèle chargé :", modele is not None)
print("Dimension :", modele.get_sentence_embedding_dimension())

Modèle chargé : True
Dimension : 384


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_26500\319309530.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimension :", modele.get_sentence_embedding_dimension())


In [17]:
print("Modèle chargé avec succès !")
print(type(modele))

Modèle chargé avec succès !
<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [ ]:
import numpy as np

print("Norme du 1er vecteur :", round(float(np.linalg.norm(embeddings[0])), 4), "(1.0 = normalisé)")
print("8 premières valeurs  :", np.round(embeddings[0][:8], 3))

Norme du 1er vecteur : 1.0 (1.0 = normalisé)
8 premières valeurs  : [ 0.025 -0.032 -0.082 -0.049  0.042 -0.028  0.02   0.09 ]


## 3. Monter la base vectorielle (ChromaDB)

On stocke pour chaque chunk : son vecteur, son texte brut et ses métadonnées (DCI, produit, type de document, fichier, page, section) — de quoi **citer la source** (fichier + page) dans la réponse finale. La base est reconstruite à chaque exécution : pas de doublons.

In [ ]:
import chromadb

VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass  # la collection n'existait pas encore

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine", "embedding_model": EMBEDDING_MODEL},
)

for i in range(0, len(chunks), 100):
    lot = chunks[i:i + 100]
    collection.add(
        ids=[c["chunk_id"] for c in lot],
        embeddings=embeddings[i:i + 100].tolist(),
        documents=[c["text"] for c in lot],
        metadatas=[{"dci": c["dci"], "produit": c["produit"], "doc_type": c["doc_type"],
                    "fichier": c["fichier"], "page": c["page"], "section": c["section"]} for c in lot],
    )

print(f"Base vectorielle prête : {VECTOR_DB_DIR.relative_to(ROOT)}  ({collection.count()} vecteurs)")

Base vectorielle prête : data\vectordb  (4411 vecteurs)


## 4. Test rapide : recherche par le sens

`chercher()` transforme la question en vecteur (préfixe `query: `) et retourne les chunks les plus proches. Les questions viennent de `docs/evaluation/questions_llm.md`.

In [19]:
def chercher(question, k=TOP_K):
    emb = modele.encode([query_prefix() + question], normalize_embeddings=True)
    res = collection.query(query_embeddings=emb.tolist(), n_results=k)
    for rang, (doc, meta, dist) in enumerate(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]), 1):
        print(f"{rang}. similarité {1 - dist:.3f} | {meta['dci']} | {meta['produit'] or '-'} | "
              f"{meta['doc_type']} p.{meta['page']}")
        print("   ", " ".join(doc.split())[:200], "\n")


from compare_search import charger_questions

for q in charger_questions()[:3]:          # changez [:3] pour tester d'autres questions
    print("=" * 100)
    print(q["id"], "—", q["texte"], "\n")
    chercher(q["texte"], k=3)

Q1 — Quelle est la liste des formes pharmaceutiques et des dosages de l’association amoxicilline + acide clavulanique répertoriés dans la nomenclature pharmaceutique algérienne consultée ? 

1. similarité 0.898 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.18
    Patients insuffisants hépatiques L’association amoxicilline/acide clavulanique doit être utilisée avec prudence chez les patients insuffisants hépatiques et la fonction hépatique doit être surveillée  

2. similarité 0.896 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.2
    concernant l’utilisation appropriée des antibactériens. 4.2. Posologie et mode d'administration Posologie Les doses sont exprimées en quantité d’amoxicilline/acide clavulanique, sauf lorsqu’elles sont 

3. similarité 0.896 | AMOXICILLINE + ACIDE CLAVULANIQUE | AMOXICILLINE_ACIDE_CLAVULANIQUE_ALMUS | RCP p.18
    à des hommes et des femmes sains, le sexe n’a pas d’incidence significa

## 5. Étape suivante : comparer avec la recherche par mots-clés

Depuis un terminal, à la racine du projet :

```
python src/compare_search.py
```

Le script pose **toutes les questions** de `docs/evaluation/questions_llm.md` aux deux méthodes (BM25 par mots-clés et recherche sémantique), sans réponse de référence, et écrit le rapport `docs/evaluation/comparaison_recherche.md` (extraits retrouvés, médicaments retrouvés, chunks communs aux deux méthodes).